# 27. Choosing where to run, across vendors

Every vendor's own tooling will tell you how your circuit does on that vendor's hardware. None of
them will tell you that the same circuit is better value somewhere else. `qbc when` answers the
question a vendor SDK structurally cannot: **of everything I can reach, where should this run?**

It ranks by predicted fidelity per dollar, which is the number that actually decides a shot budget.
Fidelity alone picks the best device; cost alone picks the cheapest; the ratio picks the one worth
paying for.

### The honest part

Ranking across vendors creates a trap. Only IBM backends are validated against real hardware here.
The other adapters exist and produce numbers, but those numbers are model and fixture estimates. If
a table put them side by side unlabelled, a reader would reasonably assume they were the same kind
of thing.

So every row declares its own provenance in a `Data` column, and unvalidated rows say so again in
words underneath. A ranking you cannot audit is worth nothing, so the tool tells you which of its
own numbers to distrust.

In [1]:
from pathlib import Path

from qiskit import QuantumCircuit

from qb_compiler.ir.converters.qasm3_converter import to_qasm3
from qb_compiler.ir.converters.qiskit_converter import from_qiskit

ghz = QuantumCircuit(3)
ghz.h(0)
for qubit in range(2):
    ghz.cx(qubit, qubit + 1)

circuit_path = Path("ghz3.qasm")
circuit_path.write_text(to_qasm3(from_qiskit(ghz)))
print(circuit_path.read_text().strip())

OPENQASM 3.0;
include "stdgates.inc";
qubit[3] q;
h q[0];
cx q[0], q[1];
cx q[1], q[2];


## 1. The default: what has calibration data

With no backend named, `qbc when` ranks everything with a loadable calibration snapshot. Today the
bundled snapshots are IBM, so the default view is IBM only. That is a statement about what ships in
the wheel, not about what the compiler can target.

In [2]:
!qbc when ghz3.qasm

----------------------------------------------------------------
Backend    | Pred.Fid | Cost USD | Fid/$ | Trend     | Data     
----------------------------------------------------------------
ibm_torino | 0.9907   | 0.5734   | 1.73  | unknown   | validated
ibm_fez    | 0.9969   | 0.6554   | 1.52  | improving | validated
----------------------------------------------------------------
ibm_torino: Calibration trend unavailable; see trend_detail.


## 2. The cross-vendor comparison

Name backends explicitly with `-b`, repeatable, to rank across vendors. This is the view that does
not exist inside any single vendor's tooling.

Read the `Data` column before the numbers.

In [3]:
!qbc when ghz3.qasm -b ibm_fez -b ionq_aria -b iqm_garnet -b rigetti_ankaa

----------------------------------------------------------------------
Backend       | Pred.Fid | Cost USD | Fid/$  | Trend     | Data       
----------------------------------------------------------------------
ibm_fez       | 0.9969   | 0.6554   | 1.52   | improving | validated  
rigetti_ankaa | 0.9633   | 1.7336   | 0.556  | unknown   | UNVALIDATED
iqm_garnet    | 0.8995   | 2.1432   | 0.42   | unknown   | UNVALIDATED
ionq_aria     | 0.9792   | 123.1800 | 0.0079 | unknown   | UNVALIDATED
----------------------------------------------------------------------
rigetti_ankaa: Calibration trend unavailable; see trend_detail.
rigetti_ankaa: Fidelity model NOT validated on real rigetti_ankaa hardware (live-unvalidated); numbers are model/fixture estimates.
iqm_garnet: Calibration trend unavailable; see trend_detail.
iqm_garnet: Fidelity model NOT validated on real iqm_garnet hardware (live-unvalidated); numbers are model/fixture estimates.
ionq_aria: Calibration trend unavailable; see tre

Two things worth reading carefully.

**The spread is enormous.** Fidelity per dollar differs by orders of magnitude across vendors for
the same three-qubit circuit, driven mostly by per-shot price rather than by fidelity. That is the
whole argument for checking before submitting.

**Only the IBM row is a measurement.** Everything marked `UNVALIDATED` is a model estimate from an
adapter that has not been proven against that device. It is still useful, since an order-of-
magnitude cost difference survives a lot of modelling error, but it is not the same evidence as the
IBM row and the table refuses to pretend otherwise.

## 3. The advice receipt

`--json` emits the same ranking as a machine-readable receipt, schema `qb.cross_vendor_advice.v1`.
Every row carries its `validation` field, so a stored receipt is still readable months later
without the table's context. It is unsigned and community-tier, like the other `--json` receipts:
free to produce, store and diff yourself.

In [4]:
import json
import subprocess

raw = subprocess.run(
    ["qbc", "when", "ghz3.qasm", "-b", "ibm_fez", "-b", "ionq_aria", "--json"],
    capture_output=True,
    text=True,
    check=True,
).stdout
advice = json.loads(raw)

print("schema:", advice["schema"])
print("shots: ", advice["shots"])
print()
for row in advice["ranking"]:
    print(f"  {row['backend']:<12} fid/$ {str(row['fidelity_per_dollar']):>8}  data={row['validation']}")

schema: qb.cross_vendor_advice.v1
shots:  4096

  ibm_fez      fid/$   1.5211  data=validated
  ionq_aria    fid/$   0.0079  data=UNVALIDATED


## 4. From Python

The CLI is a thin wrapper over `rank_value`, so the same ranking is available directly, including
the `notes` that spell out each caveat in words.

In [5]:
from qb_compiler.windows import rank_value

rows = rank_value(ghz, backends=["ibm_fez", "ionq_aria"], n_seeds=2)
for row in rows:
    print(f"{row.backend}  validation={row.validation}")
    for note in row.notes:
        print(f"    - {note}")

ibm_fez  validation=validated
ionq_aria  validation=UNVALIDATED
    - Calibration trend unavailable; see trend_detail.
    - Fidelity model NOT validated on real ionq_aria hardware (live-unvalidated); numbers are model/fixture estimates.


## Summary

- `qbc when` ranks by fidelity per dollar across vendors, which is the comparison a single-vendor
  SDK cannot give you.
- The `Data` column states whether each number is validated on real hardware or a model estimate.
  Only IBM is validated today, and the table says so rather than letting the distinction blur.
- `--json` gives the same ranking as an unsigned advice receipt you can store and diff.

The ranking, the receipt and the CLI are free and Apache-2.0. Signing those receipts, keeping their
history, sharing them across a team and gating CI on them is QubitBoost Pro, as is the work of
turning `UNVALIDATED` rows into validated ones through per-vendor calibration adapters.

In [6]:
circuit_path.unlink(missing_ok=True)